# StringSense Complete ABSA Pipeline — Immutable Experiment Stage

This notebook consumes only the labeling datasets from the same immutable run. It trains deterministic TF-IDF baselines, performs full-corpus inference, writes versioned artifacts and produces the final run manifest.

## Resolve the canonical workbench and run ID

In [ ]:
import json
import os
import sys
from pathlib import Path

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data/archive_latest/badminton_strings_data.json').is_file():
    BASE_DIR = BASE_DIR / 'ml/nlp-workbench-latest'
if not (BASE_DIR / 'data/archive_latest/badminton_strings_data.json').is_file():
    raise FileNotFoundError('Cannot locate the canonical NLP workbench')
sys.path.insert(0, str(BASE_DIR / 'src'))

from stringsense_nlp.pipeline import run_pipeline

RUN_ID = os.environ.get('STRINGSENSE_NLP_RUN_ID')
if not RUN_ID:
    raise RuntimeError('Set STRINGSENSE_NLP_RUN_ID or use scripts/run_experiment.py')
RUN_ID

## Train, evaluate and generate versioned artifacts

In [ ]:
pipeline_result = run_pipeline(RUN_ID, BASE_DIR)
pipeline_result['summary']

## Inspect the final manifest and promotion gate

In [ ]:
with Path(pipeline_result['run_manifest_path']).open('r', encoding='utf-8') as handle:
    run_manifest = json.load(handle)

assert run_manifest['status'] == 'completed'
assert run_manifest['promotion']['status'] == 'not_promoted'
assert run_manifest['promotion']['requires_human_approval'] is True
run_manifest['promotion']

No file in this run is promoted automatically. The canonical backend workbook remains unchanged until a separate, explicit approval and comparison task.